# A short summary of the tasks and methods

In [1]:
# Requirements:

# 1. Implement and create our own image segmentation model from scratch using PyTorch or TensorFlow. This will involve designing the architecture, defining the loss function, 
# and training the model on a dataset of images with corresponding segmentation masks.

# 2. Image segmentation task using semantic segmentation with a pre-trained model (In our case segformer) and fine-tuning it on our dataset.
# Transformers follow an encoder-decoder architecture, where the encoder processes the input image and turns them into vector representations using multi-head self-attention
# and position-wise feed-forward networks.
# Decoder takes these vector representations and generates the segmented output using masked self-attention, encoder-decoder attention and feed-forward layers.

# Multi-head self-attention lets every token (pixel/patch) look at every other token and decide how important they are. 
# It turns each input vector into Query, Key, and Value vectors (Q, K, V). Query - what am i looking for? Key - what do i have? Value - what do i output?
# Then attention scores are calculated by taking the dot product of the Query with all Keys, dividing by the square root of the dimension of the Key vectors, 
# and applying a softmax function to get weights. Weights are values between 0 and 1 that indicate how much attention each token should pay to every other token.
# One head isn't enough to capture all the relationships in the data, so we use multiple heads to allow the model to attend to different parts of the input simultaneously.
# "This pixel belongs to a person, this pixel belongs to a car, this pixel belongs to a dog" - the model can learn to focus on different features for each class.

# Chosen classes: 
# 1. Person 
# 2. Car 
# 3. Dog

# We will use the Hugging Face Transformers library to load a pre-trained SegFormer model and fine-tune it on our dataset.


# 1. Implementing our own image segmentation model from scratch using PyTorch:

In [2]:
# 1. Creating a U-Net architecture for image segmentation:
# U-net is a convolutional neural network architecture that is widely used for image segmentation tasks, where each pixel in an image is classified into a specific category.
# The architecture consists of an encoder, which exctracts features from the input image using 3x3 convolutional layers followed by ReLU activation and max pooling.
# 1. The kernel extracts features, 2. ReLU adds non-linearity, 3. Max pooling reduces the spatial dimensions of the feature maps while retaining important information. 
# 3. Max pooling reduces from a 2x2 matrix to a single value, taking only the biggest value in the 2x2 area, which helps to reduce the computational load and capture 
# the most important features.
# Decoder upsamples the feature maps back to the original image size using transposed convolutional layers, which are also known as deconvolutional layers.
# Combines features from the encoder with the upsampled features in the decoder using skip connections, which helps to preserve spatial information and improve 
# segmentation accuracy.
# Skip connections - the output of one layer is added to the output of a deeper layer, bypassing intermediate layers. This helps mitigate gradient vanishing problems.
# It also allows the model to reuse earlier features, which allows fine details from early layers to be combined with high-level features.
# Output layer uses a softmax activation function to produce a probability distribution over the classes for each pixel, allowing the model to classify each pixel into
# one of the predefined classes.
# Most typical loss function for image segmentation is the cross-entropy loss, which measures the difference between the predicted class probabilities and the true class 
# labels for each pixel.

# Install Needed libraries

In [3]:
# Install PyTorch
%pip install torch torchvision

# Install other necessary libraries
%pip install numpy matplotlib tqdm


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Implementing U-net in Pytorch

In [4]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 1, 1, bias = False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace = True),
            nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias = False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace = True)
        )

    def forward(self, x):
        return self.conv(x)
    
class UNet(nn.Module):
    def __init__(
            self, in_channels = 3, out_channels = 3, features = [64, 128, 256, 512]
    ):
        super().__init__()
        self.ups = nn.ModuleList()
        self.downs = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size = 2, stride = 2)

        # Down part of UNET
        for feature in features:
            self.downs.append(DoubleConv(in_channels, feature))
            in_channels = feature

        # Bottleneck
        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)

        # Up part of UNET
        for feature in reversed(features):
            self.ups.append(
                nn.ConvTranspose2d(
                    feature * 2, feature, kernel_size = 2, stride = 2
                )
            )
            self.ups.append(DoubleConv(feature * 2, feature))

        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size = 1)

    def forward(self, x):
        skip_connections = []

        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]

        for idx in range(0, len(self.ups), 2):
            x = self.ups[idx](x)
            skip_connection = skip_connections[idx // 2]

            if x.shape != skip_connection.shape:
                x = nn.functional.interpolate(
                    x, size = skip_connection.shape[2:], mode = "bilinear", align_corners = True
                    )
                
            concat_skip = torch.cat((skip_connection, x), dim = 1)
            x = self.ups[idx + 1](concat_skip)

        return self.final_conv(x)

# Training the U-Net Model



In [6]:
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Sequence, Tuple

import numpy as np
import torch
from PIL import Image
from torch.utils.data import ConcatDataset, DataLoader, Dataset, Subset
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

FAST_MODE = True
CACHE_DATA = True
FINAL_TARGET_PER_CLASS = 400
TEST_TOTAL = 100
VAL_FRACTION = 0.15
SEED = 42
CHECKPOINT_DIR = Path.cwd() / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["background", "person", "car", "dog"]
NUM_CLASSES = len(CLASS_NAMES)
ID_PERSON, ID_CAR, ID_DOG = 1, 2, 3

CPU_CORES = os.cpu_count() or 8
TRAIN_TARGET_PER_CLASS = FINAL_TARGET_PER_CLASS
IMAGE_SIZE = (128, 128) if FAST_MODE else (256, 256)
BATCH_SIZE = 24 if device.type == "cuda" else 8
NUM_WORKERS = 2 if device.type == "cuda" else 0

if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass
else:
    torch.set_num_threads(min(8, CPU_CORES))
    try:
        torch.set_num_interop_threads(1)
    except RuntimeError:
        pass

def seed_everything(seed: int = 42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

@dataclass(frozen=True)
class SegAugmentConfig:
    hflip_p: float = 0.5
    affine_p: float = 0.0
    degrees: float = 0.0
    translate: float = 0.0
    scale: Tuple[float, float] = (1.0, 1.0)
    brightness: Tuple[float, float] = (1.0, 1.0)
    contrast: Tuple[float, float] = (1.0, 1.0)
    saturation: Tuple[float, float] = (1.0, 1.0)

class CachedBinarySegFolderDataset(Dataset):
    """Loads binary masks and maps foreground pixels to one semantic class id."""

    def __init__(
        self,
        images_dir: Path,
        masks_dir: Path,
        *,
        foreground_class_id: int,
        size: Tuple[int, int],
        cache: bool = True,
        image_exts: Sequence[str] = (".jpg", ".jpeg", ".png", ".webp"),
        mask_exts: Sequence[str] = (".png", ".jpg", ".jpeg", ".gif"),
        mask_prefix: str = "",
        mask_suffix: str = "",
    ):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.foreground_class_id = int(foreground_class_id)
        self.size = tuple(size)
        self.cache = bool(cache)
        self.image_exts = tuple(e.lower() for e in image_exts)
        self.mask_exts = tuple(e.lower() for e in mask_exts)
        self.mask_prefix = str(mask_prefix)
        self.mask_suffix = str(mask_suffix)

        images = [
            p
            for p in sorted(self.images_dir.iterdir())
            if p.is_file() and p.suffix.lower() in self.image_exts
        ]
        if not images:
            raise FileNotFoundError(f"No images found in {self.images_dir}")

        self.pairs = []
        for img_path in images:
            found = None
            for ext in self.mask_exts:
                candidate = self.masks_dir / f"{self.mask_prefix}{img_path.stem}{self.mask_suffix}{ext}"
                if candidate.exists():
                    found = candidate
                    break
            if found is not None:
                self.pairs.append((img_path, found))

        if not self.pairs:
            example = images[0].stem
            raise FileNotFoundError(
                f"No matching masks found in {self.masks_dir}. Tried names like "
                f"'{self.mask_prefix}{example}{self.mask_suffix}<ext>'."
            )

        self.samples = None
        if self.cache:
            self.samples = [self._load_pair(img_path, mask_path) for img_path, mask_path in self.pairs]

    def _load_pair(self, img_path: Path, mask_path: Path):
        img = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        img = TF.resize(img, self.size, interpolation=InterpolationMode.BILINEAR)
        mask = TF.resize(mask, self.size, interpolation=InterpolationMode.NEAREST)

        img_t = TF.to_tensor(img)
        mask_np = np.array(mask, dtype=np.uint8)
        out = np.zeros_like(mask_np, dtype=np.uint8)
        out[mask_np > 0] = np.uint8(self.foreground_class_id)
        mask_t = torch.from_numpy(out)
        return img_t.contiguous(), mask_t.contiguous()

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        if self.samples is None:
            img_t, mask_t = self._load_pair(*self.pairs[idx])
        else:
            img_t, mask_t = self.samples[idx]
        return img_t.clone(), mask_t.clone()

class AugmentedDataset(Dataset):
    def __init__(self, base: Dataset, cfg: Optional[SegAugmentConfig] = None):
        self.base = base
        self.cfg = cfg

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, mask = self.base[idx]
        if self.cfg is None:
            return img, mask

        if torch.rand(1).item() < self.cfg.hflip_p:
            img = torch.flip(img, dims=[2])
            mask = torch.flip(mask, dims=[1])

        if torch.rand(1).item() < self.cfg.affine_p:
            max_dx = self.cfg.translate * img.shape[2]
            max_dy = self.cfg.translate * img.shape[1]
            translate = [
                int(round(torch.empty(1).uniform_(-max_dx, max_dx).item())),
                int(round(torch.empty(1).uniform_(-max_dy, max_dy).item())),
            ]
            angle = float(torch.empty(1).uniform_(-self.cfg.degrees, self.cfg.degrees).item())
            scale = float(torch.empty(1).uniform_(*self.cfg.scale).item())

            img = TF.affine(
                img,
                angle=angle,
                translate=translate,
                scale=scale,
                shear=[0.0, 0.0],
                interpolation=InterpolationMode.BILINEAR,
                fill=0.0,
            )
            mask = TF.affine(
                mask.unsqueeze(0).float(),
                angle=angle,
                translate=translate,
                scale=scale,
                shear=[0.0, 0.0],
                interpolation=InterpolationMode.NEAREST,
                fill=0.0,
            ).squeeze(0).to(torch.uint8)

        b = float(torch.empty(1).uniform_(*self.cfg.brightness).item())
        c = float(torch.empty(1).uniform_(*self.cfg.contrast).item())
        s = float(torch.empty(1).uniform_(*self.cfg.saturation).item())
        img = TF.adjust_brightness(img, b)
        img = TF.adjust_contrast(img, c)
        img = TF.adjust_saturation(img, s)
        return img.clamp(0.0, 1.0).contiguous(), mask.contiguous()

class OversampledDataset(Dataset):
    """Deterministic oversampling to a fixed size."""

    def __init__(self, base: Dataset, length: int, seed: int):
        self.base = base
        self.length = int(length)
        if len(base) == 0:
            raise ValueError("Base dataset is empty")
        g = torch.Generator().manual_seed(int(seed))
        self.indices = torch.randint(0, len(base), (self.length,), generator=g).tolist()

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        return self.base[self.indices[idx]]

def split_train_val_test(n: int, n_test: int, val_fraction: float, seed: int):
    if n_test >= n:
        raise ValueError(f"n_test must be < n. Got n_test={n_test}, n={n}")
    g = torch.Generator().manual_seed(int(seed))
    perm = torch.randperm(n, generator=g).tolist()
    test_idx = perm[:n_test]
    remaining = perm[n_test:]
    n_val = max(1, int(round(len(remaining) * val_fraction)))
    n_val = min(n_val, len(remaining) - 1)
    val_idx = remaining[:n_val]
    train_idx = remaining[n_val:]
    return train_idx, val_idx, test_idx

DATA_ROOT = Path.cwd() / "data"
CAR_CFG = SegAugmentConfig(
    hflip_p=0.5,
    affine_p=0.15,
    degrees=6.0,
    translate=0.04,
    scale=(0.95, 1.05),
    brightness=(0.92, 1.08),
    contrast=(0.92, 1.08),
    saturation=(0.92, 1.08),
)
PERSON_DOG_CFG = SegAugmentConfig(
    hflip_p=0.5,
    affine_p=0.35,
    degrees=12.0,
    translate=0.08,
    scale=(0.9, 1.1),
    brightness=(0.85, 1.15),
    contrast=(0.85, 1.15),
    saturation=(0.85, 1.15),
)

car_base = CachedBinarySegFolderDataset(
    DATA_ROOT / "car" / "images",
    DATA_ROOT / "car" / "masks",
    foreground_class_id=ID_CAR,
    size=IMAGE_SIZE,
    cache=CACHE_DATA,
)
person_base = CachedBinarySegFolderDataset(
    DATA_ROOT / "person" / "images",
    DATA_ROOT / "person" / "masks",
    foreground_class_id=ID_PERSON,
    size=IMAGE_SIZE,
    cache=CACHE_DATA,
)
dog_base = CachedBinarySegFolderDataset(
    DATA_ROOT / "dog" / "images",
    DATA_ROOT / "dog" / "masks",
    foreground_class_id=ID_DOG,
    size=IMAGE_SIZE,
    cache=CACHE_DATA,
    mask_prefix="annotated_",
)

base_sizes = {"car": len(car_base), "person": len(person_base), "dog": len(dog_base)}
print("Base sizes:", base_sizes)

test_base = TEST_TOTAL // 3
remainder = TEST_TOTAL - 3 * test_base
test_counts = {"car": test_base, "person": test_base, "dog": test_base}
for key in ("car", "person", "dog")[:remainder]:
    test_counts[key] += 1
for key in list(test_counts.keys()):
    test_counts[key] = min(test_counts[key], base_sizes[key] - 2)
print("Test counts per class:", test_counts, "total=", sum(test_counts.values()))

car_train_idx, car_val_idx, car_test_idx = split_train_val_test(base_sizes["car"], test_counts["car"], VAL_FRACTION, SEED)
person_train_idx, person_val_idx, person_test_idx = split_train_val_test(base_sizes["person"], test_counts["person"], VAL_FRACTION, SEED + 1)
dog_train_idx, dog_val_idx, dog_test_idx = split_train_val_test(base_sizes["dog"], test_counts["dog"], VAL_FRACTION, SEED + 2)

car_train = OversampledDataset(AugmentedDataset(Subset(car_base, car_train_idx), CAR_CFG), TRAIN_TARGET_PER_CLASS, seed=SEED + 10)
person_train = OversampledDataset(AugmentedDataset(Subset(person_base, person_train_idx), PERSON_DOG_CFG), TRAIN_TARGET_PER_CLASS, seed=SEED + 20)
dog_train = OversampledDataset(AugmentedDataset(Subset(dog_base, dog_train_idx), PERSON_DOG_CFG), TRAIN_TARGET_PER_CLASS, seed=SEED + 30)

train_ds = ConcatDataset([car_train, person_train, dog_train])
val_ds = ConcatDataset([
    Subset(car_base, car_val_idx),
    Subset(person_base, person_val_idx),
    Subset(dog_base, dog_val_idx),
])
test_ds = ConcatDataset([
    Subset(car_base, car_test_idx),
    Subset(person_base, person_test_idx),
    Subset(dog_base, dog_test_idx),
])

print("Train size:", len(train_ds), "(balanced with augmentation)")
print("Val size  :", len(val_ds), "(real images, no augmentation)")
print("Test size :", len(test_ds), "(unseen images, no augmentation)")
print("CPU threads:", torch.get_num_threads(), "| workers:", NUM_WORKERS, "| train target/class:", TRAIN_TARGET_PER_CLASS)

loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)
if NUM_WORKERS > 0:
    loader_kwargs["prefetch_factor"] = 2

train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_ds, shuffle=False, **loader_kwargs)

xb, yb = next(iter(train_loader))
print("train batch image shape:", xb.shape, xb.dtype)
print("train batch mask shape :", yb.shape, yb.dtype)
print("train mask classes     :", torch.unique(yb).tolist())


Using device: cpu
Base sizes: {'car': 256, 'person': 180, 'dog': 180}
Test counts per class: {'car': 34, 'person': 33, 'dog': 33} total= 100
Train size: 1200 (balanced with augmentation)
Val size  : 77 (real images, no augmentation)
Test size : 100 (unseen images, no augmentation)
CPU threads: 8 | workers: 0 | train target/class: 400
train batch image shape: torch.Size([8, 3, 128, 128]) torch.float32
train batch mask shape : torch.Size([8, 128, 128]) torch.uint8
train mask classes     : [0, 1, 2, 3]


In [7]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

USE_AMP = (device.type == "cuda")
USE_COMPILE = False
FEATURES = [32, 64, 128, 256, 512]
EPOCHS = 10
LEARNING_RATE = 3e-4
EARLY_STOPPING_PATIENCE = 5
MIN_IMPROVEMENT = 1e-3
VALIDATE_EVERY = 1

model = UNet(in_channels=3, out_channels=NUM_CLASSES, features=FEATURES)
model = model.to(device, memory_format=torch.channels_last)

if USE_COMPILE:
    try:
        model = torch.compile(model)
        print("torch.compile enabled")
    except Exception as e:
        print("torch.compile not available:", e)

@torch.no_grad()
def estimate_class_weights(loader, num_classes: int):
    counts = torch.zeros(num_classes, dtype=torch.float64)
    for _, yb in loader:
        counts += torch.bincount(yb.view(-1), minlength=num_classes).to(torch.float64)
    weights = counts.sum() / (num_classes * counts.clamp_min(1.0))
    weights = weights / weights.mean()
    return weights.to(device=device, dtype=torch.float32)

class SegmentationLoss(nn.Module):
    def __init__(self, ce_weight: torch.Tensor, dice_weight: float = 0.5):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=ce_weight)
        self.dice_weight = float(dice_weight)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor):
        ce = self.ce(logits, targets)
        probs = torch.softmax(logits, dim=1)
        target_1h = F.one_hot(targets, num_classes=logits.shape[1]).permute(0, 3, 1, 2).float()

        probs_fg = probs[:, 1:]
        target_fg = target_1h[:, 1:]
        dims = (0, 2, 3)
        intersection = (probs_fg * target_fg).sum(dim=dims)
        denominator = probs_fg.sum(dim=dims) + target_fg.sum(dim=dims)
        dice = (2.0 * intersection + 1e-6) / (denominator + 1e-6)
        dice_loss = 1.0 - dice.mean()
        return ce + self.dice_weight * dice_loss

class_weights = estimate_class_weights(train_loader, NUM_CLASSES)
print("Class weights:", {name: round(float(w), 3) for name, w in zip(CLASS_NAMES, class_weights)})

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
)
criterion = SegmentationLoss(class_weights, dice_weight=0.5)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

@torch.no_grad()
def evaluate(model, loader, num_classes: int):
    model.eval()
    total_loss = 0.0
    conf = torch.zeros((num_classes, num_classes), device=device, dtype=torch.int64)

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True, memory_format=torch.channels_last)
        yb = yb.to(device, dtype=torch.long, non_blocking=True)

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(xb)
            loss = criterion(logits, yb)

        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        idx = num_classes * yb.view(-1) + preds.view(-1)
        conf += torch.bincount(idx, minlength=num_classes * num_classes).reshape(num_classes, num_classes)

    conf = conf.float()
    diag = conf.diag()
    recall = diag / (conf.sum(1) + 1e-7)
    precision = diag / (conf.sum(0) + 1e-7)
    f1 = 2.0 * precision * recall / (precision + recall + 1e-7)
    denom = conf.sum(1) + conf.sum(0) - diag + 1e-7
    iou = diag / denom
    pixel_acc = (diag.sum() / (conf.sum() + 1e-7)).item()

    return {
        "loss": total_loss / max(1, len(loader)),
        "pixel_acc": pixel_acc,
        "miou_all": float(iou.mean().item()),
        "miou_fg": float(iou[1:].mean().item()),
        "iou_per_class": iou.detach().cpu().numpy(),
        "precision_per_class": precision.detach().cpu().numpy(),
        "recall_per_class": recall.detach().cpu().numpy(),
        "f1_per_class": f1.detach().cpu().numpy(),
    }

def train_one_epoch(model, loader):
    model.train()
    running_loss = 0.0

    for xb, yb in tqdm(loader, desc="train", leave=False):
        xb = xb.to(device, non_blocking=True, memory_format=torch.channels_last)
        yb = yb.to(device, dtype=torch.long, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(xb)
            loss = criterion(logits, yb)

        if USE_AMP:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        running_loss += loss.item()

    return running_loss / max(1, len(loader))

best_val_miou = -float("inf")
best_epoch = 0
best_state = None
epochs_without_improvement = 0
best_local_checkpoint_path = CHECKPOINT_DIR / "unet_best_local.pt"

print("Model features:", FEATURES)
print("Image size:", IMAGE_SIZE, "| batch size:", BATCH_SIZE, "| AMP:", USE_AMP)

last_val_metrics = None

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader)
    ran_validation = (epoch % VALIDATE_EVERY == 0) or (epoch == EPOCHS)
    if ran_validation:
        val_metrics = evaluate(model, val_loader, NUM_CLASSES)
        last_val_metrics = val_metrics
        scheduler.step(val_metrics["miou_fg"])

        if val_metrics["miou_fg"] > best_val_miou + MIN_IMPROVEMENT:
            best_val_miou = val_metrics["miou_fg"]
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
    else:
        val_metrics = last_val_metrics

    current_lr = optimizer.param_groups[0]["lr"]
    if ran_validation and val_metrics is not None:
        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"lr={current_lr:.2e} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} | "
            f"pixAcc={val_metrics['pixel_acc']:.3f} | "
            f"mIoU_fg={val_metrics['miou_fg']:.3f}"
        )
    else:
        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"lr={current_lr:.2e} | "
            f"train_loss={train_loss:.4f} | "
            f"validation=skipped"
        )

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping triggered after {epoch} epochs.")
        break

if best_state is not None:
    model.load_state_dict(best_state)
    torch.save(
        {
            "model_state_dict": best_state,
            "features": FEATURES,
            "num_classes": NUM_CLASSES,
            "class_names": CLASS_NAMES,
            "image_size": IMAGE_SIZE,
            "best_epoch": best_epoch,
            "best_val_miou": best_val_miou,
        },
        best_local_checkpoint_path,
    )
    print("Saved best local checkpoint to:", best_local_checkpoint_path)

print("Evaluating best model on the held-out test set...")
test_metrics = evaluate(model, test_loader, NUM_CLASSES)
print("Best epoch selected from validation:", best_epoch)
print(
    f"TEST | loss={test_metrics['loss']:.4f} | "
    f"pixAcc={test_metrics['pixel_acc']:.3f} | "
    f"mIoU_fg={test_metrics['miou_fg']:.3f}"
)

for idx, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_name:>10} | "
        f"IoU={test_metrics['iou_per_class'][idx]:.3f} | "
        f"Precision={test_metrics['precision_per_class'][idx]:.3f} | "
        f"Recall={test_metrics['recall_per_class'][idx]:.3f} | "
        f"F1={test_metrics['f1_per_class'][idx]:.3f}"
    )


Class weights: {'background': 0.197, 'person': 1.012, 'car': 1.801, 'dog': 0.99}
Model features: [32, 64, 128, 256, 512]
Image size: (128, 128) | batch size: 8 | AMP: False


Epoch 01/10 | lr=3.00e-04 | train_loss=1.2820 | val_loss=1.1107 | pixAcc=0.571 | mIoU_fg=0.351


Epoch 02/10 | lr=3.00e-04 | train_loss=1.0500 | val_loss=1.0080 | pixAcc=0.738 | mIoU_fg=0.444


Epoch 03/10 | lr=3.00e-04 | train_loss=0.9912 | val_loss=1.0421 | pixAcc=0.705 | mIoU_fg=0.409


Epoch 04/10 | lr=3.00e-04 | train_loss=0.9039 | val_loss=0.9238 | pixAcc=0.750 | mIoU_fg=0.483


...

Epoch 06/10 | lr=3.00e-04 | train_loss=0.8115 | val_loss=0.7769 | pixAcc=0.758 | mIoU_fg=0.563


Epoch 07/10 | lr=3.00e-04 | train_loss=0.8346 | val_loss=0.8027 | pixAcc=0.801 | mIoU_fg=0.593


Epoch 08/10 | lr=3.00e-04 | train_loss=0.8116 | val_loss=0.9141 | pixAcc=0.795 | mIoU_fg=0.515


Epoch 09/10 | lr=3.00e-04 | train_loss=0.7434 | val_loss=0.8085 | pixAcc=0.751 | mIoU_fg=0.555


Epoch 10/10 | lr=1.50e-04 | train_loss=0.7190 | val_loss=0.9347 | pixAcc=0.709 | mIoU_fg=0.495
Saved best local checkpoint to: c:\Users\kipra\OneDrive\Dokumente\New project\checkpoints\unet_best_local.pt
Evaluating best model on the held-out test set...
Best epoch selected from validation: 7
TEST | loss=0.8868 | pixAcc=0.757 | mIoU_fg=0.566
background | IoU=0.709 | Precision=0.949 | Recall=0.737 | F1=0.830
    person | IoU=0.368 | Precision=0.415 | Recall=0.767 | F1=0.539
       car | IoU=0.897 | Precision=0.905 | Recall=0.989 | F1=0.946
       dog | IoU=0.433 | Precision=0.520 | Recall=0.720 | F1=0.604


## External Open Images evaluation

This section is for the **final external test** on 100 unseen Open Images images containing at least one of the classes: `Person`, `Car`, or `Dog`.


In [ ]:
# External evaluation on 100 unseen Open Images validation images
# %pip install fiftyone

import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode

import fiftyone.zoo as foz
from fiftyone.utils.labels import objects_to_segmentations

OI_CLASSES = ["Person", "Car", "Dog"]
OI_MASK_TARGETS = {1: "Person", 2: "Car", 3: "Dog"}
OI_MAX_SAMPLES = 100

oi_dataset = foz.load_zoo_dataset(
    "open-images-v7",
    split="validation",
    label_types=["segmentations"],
    classes=OI_CLASSES,
    only_matching=True,
    max_samples=OI_MAX_SAMPLES,
    shuffle=True,
    seed=SEED,
    dataset_name="open-images-v7-person-car-dog-seg-100",
)

instance_field = None
for candidate in ("ground_truth", "detections", "segmentations"):
    try:
        objects_to_segmentations(
            oi_dataset,
            candidate,
            "gt_semantic",
            mask_targets=OI_MASK_TARGETS,
            overwrite=True,
            progress=True,
        )
        instance_field = candidate
        break
    except Exception:
        continue

if instance_field is None:
    raise RuntimeError(
        "Could not find the Open Images instance-segmentation field. "
        "Inspect `oi_dataset.get_field_schema()` and update the candidate list."
    )

print("Open Images masks converted from field:", instance_field)
print("Downloaded samples:", len(oi_dataset))

class OpenImagesSemanticDataset(Dataset):
    def __init__(self, samples, size):
        self.samples = list(samples.iter_samples(progress=True))
        self.size = tuple(size)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img = Image.open(sample.filepath).convert("RGB")
        img = TF.resize(img, self.size, interpolation=InterpolationMode.BILINEAR)
        img_t = TF.to_tensor(img)

        seg = sample["gt_semantic"]
        mask_np = seg.get_mask()
        if mask_np is None:
            raise RuntimeError(f"No semantic mask found for sample: {sample.filepath}")

        mask_t = torch.from_numpy(mask_np.astype("uint8"))
        mask_t = TF.resize(
            mask_t.unsqueeze(0).float(),
            self.size,
            interpolation=InterpolationMode.NEAREST,
        ).squeeze(0).to(torch.long)

        return img_t.contiguous(), mask_t.contiguous()

oi_ds = OpenImagesSemanticDataset(oi_dataset, IMAGE_SIZE)
oi_loader = DataLoader(
    oi_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

oi_metrics = evaluate(model, oi_loader, NUM_CLASSES)
print(
    f"OPENIMAGES-100 | loss={oi_metrics['loss']:.4f} | "
    f"pixAcc={oi_metrics['pixel_acc']:.3f} | "
    f"mIoU_fg={oi_metrics['miou_fg']:.3f}"
)

for idx, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_name:>10} | "
        f"IoU={oi_metrics['iou_per_class'][idx]:.3f} | "
        f"Precision={oi_metrics['precision_per_class'][idx]:.3f} | "
        f"Recall={oi_metrics['recall_per_class'][idx]:.3f} | "
        f"F1={oi_metrics['f1_per_class'][idx]:.3f}"
    )


c:\Users\kipra\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Necessary images already downloaded
Existing download of split 'validation' is sufficient
Loading existing dataset 'open-images-v7-person-car-dog-seg-100'. To reload from disk, either delete the existing dataset or provide a custom `dataset_name` to use
 100% |█████████████████| 100/100 [1.8s elapsed, 0s remaining, 51.6 samples/s]         
Open Images masks converted from field: ground_truth
Downloaded samples: 100
 100% |█████████████████| 100/100 [397.2ms elapsed, 0s remaining, 251.8 samples/s]      
OPENIMAGES-100 | loss=2.6196 | pixAcc=0.492 | mIoU_fg=0.099
background | IoU=0.482 | Precision=0.928 | Recall=0.501 | F1=0.651
    person | IoU=0.052 | Precision=0.053 | Recall=0.644 | F1=0.099
       car | IoU=0.000 | Precision=0.000 | Recall=0.000 | F1=0.000
       dog | IoU=0.246 | Precision=0.273 | Recall=0.711 | F1=0.394


## Final training on all local data + external Open Images test

This will:
1. Rebuild a training set from **the local dataset**
2. Retrain a fresh model for the selected number of epochs
3. Download and evaluate on **100 unseen Open Images validation images**


In [9]:
# Final model: train on all local data, then evaluate on unseen Open Images images

FINAL_EPOCHS = best_epoch if "best_epoch" in globals() and best_epoch > 0 else 10
final_checkpoint_path = CHECKPOINT_DIR / "unet_final_presentation.pt"
print("Final training epochs:", FINAL_EPOCHS)

full_car_train = OversampledDataset(AugmentedDataset(car_base, CAR_CFG), FINAL_TARGET_PER_CLASS, seed=SEED + 110)
full_person_train = OversampledDataset(AugmentedDataset(person_base, PERSON_DOG_CFG), FINAL_TARGET_PER_CLASS, seed=SEED + 120)
full_dog_train = OversampledDataset(AugmentedDataset(dog_base, PERSON_DOG_CFG), FINAL_TARGET_PER_CLASS, seed=SEED + 130)
final_train_ds = ConcatDataset([full_car_train, full_person_train, full_dog_train])

final_loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)
if NUM_WORKERS > 0:
    final_loader_kwargs["prefetch_factor"] = 2

final_train_loader = DataLoader(final_train_ds, shuffle=True, **final_loader_kwargs)

final_model = UNet(in_channels=3, out_channels=NUM_CLASSES, features=FEATURES)
final_model = final_model.to(device, memory_format=torch.channels_last)

final_optimizer = torch.optim.AdamW(final_model.parameters(), lr=LEARNING_RATE)
final_class_weights = estimate_class_weights(final_train_loader, NUM_CLASSES)
final_criterion = SegmentationLoss(final_class_weights, dice_weight=0.5)
final_scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

def train_one_epoch_final(model, loader):
    model.train()
    running_loss = 0.0

    for xb, yb in tqdm(loader, desc="final-train", leave=False):
        xb = xb.to(device, non_blocking=True, memory_format=torch.channels_last)
        yb = yb.to(device, dtype=torch.long, non_blocking=True)

        final_optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(xb)
            loss = final_criterion(logits, yb)

        if USE_AMP:
            final_scaler.scale(loss).backward()
            final_scaler.step(final_optimizer)
            final_scaler.update()
        else:
            loss.backward()
            final_optimizer.step()

        running_loss += loss.item()

    return running_loss / max(1, len(loader))

for epoch in range(1, FINAL_EPOCHS + 1):
    final_train_loss = train_one_epoch_final(final_model, final_train_loader)
    print(f"Final epoch {epoch:02d}/{FINAL_EPOCHS} | train_loss={final_train_loss:.4f}")

torch.save(
    {
        "model_state_dict": final_model.state_dict(),
        "features": FEATURES,
        "num_classes": NUM_CLASSES,
        "class_names": CLASS_NAMES,
        "image_size": IMAGE_SIZE,
        "epochs": FINAL_EPOCHS,
        "train_target_per_class": FINAL_TARGET_PER_CLASS,
    },
    final_checkpoint_path,
)
print("Saved final checkpoint to:", final_checkpoint_path)

if "oi_loader" not in globals():
    import fiftyone.zoo as foz
    from fiftyone.utils.labels import objects_to_segmentations

    OI_CLASSES = ["Person", "Car", "Dog"]
    OI_MASK_TARGETS = {1: "Person", 2: "Car", 3: "Dog"}
    OI_MAX_SAMPLES = 100

    oi_dataset = foz.load_zoo_dataset(
        "open-images-v7",
        split="validation",
        label_types=["segmentations"],
        classes=OI_CLASSES,
        only_matching=True,
        max_samples=OI_MAX_SAMPLES,
        shuffle=True,
        seed=SEED,
        dataset_name="open-images-v7-person-car-dog-seg-100",
    )

    instance_field = None
    for candidate in ("ground_truth", "detections", "segmentations"):
        try:
            objects_to_segmentations(
                oi_dataset,
                candidate,
                "gt_semantic",
                mask_targets=OI_MASK_TARGETS,
                overwrite=True,
                progress=True,
            )
            instance_field = candidate
            break
        except Exception:
            continue

    if instance_field is None:
        raise RuntimeError(
            "Could not find the Open Images instance-segmentation field. "
            "Inspect `oi_dataset.get_field_schema()` and update the candidate list."
        )

    class OpenImagesSemanticDataset(Dataset):
        def __init__(self, samples, size):
            self.samples = list(samples.iter_samples(progress=True))
            self.size = tuple(size)

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, idx):
            sample = self.samples[idx]
            img = Image.open(sample.filepath).convert("RGB")
            img = TF.resize(img, self.size, interpolation=InterpolationMode.BILINEAR)
            img_t = TF.to_tensor(img)

            seg = sample["gt_semantic"]
            mask_np = seg.get_mask()
            if mask_np is None:
                raise RuntimeError(f"No semantic mask found for sample: {sample.filepath}")

            mask_t = torch.from_numpy(mask_np.astype("uint8"))
            mask_t = TF.resize(
                mask_t.unsqueeze(0).float(),
                self.size,
                interpolation=InterpolationMode.NEAREST,
            ).squeeze(0).to(torch.long)
            return img_t.contiguous(), mask_t.contiguous()

    oi_ds = OpenImagesSemanticDataset(oi_dataset, IMAGE_SIZE)
    oi_loader = DataLoader(
        oi_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=(device.type == "cuda"),
    )

original_criterion = criterion
criterion = final_criterion
oi_metrics = evaluate(final_model, oi_loader, NUM_CLASSES)
criterion = original_criterion

print(
    f"FINAL OPENIMAGES-100 | loss={oi_metrics['loss']:.4f} | "
    f"pixAcc={oi_metrics['pixel_acc']:.3f} | "
    f"mIoU_fg={oi_metrics['miou_fg']:.3f}"
)
for idx, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_name:>10} | "
        f"IoU={oi_metrics['iou_per_class'][idx]:.3f} | "
        f"Precision={oi_metrics['precision_per_class'][idx]:.3f} | "
        f"Recall={oi_metrics['recall_per_class'][idx]:.3f} | "
        f"F1={oi_metrics['f1_per_class'][idx]:.3f}"
    )


Final training epochs: 7


Final epoch 01/7 | train_loss=1.3341


Final epoch 02/7 | train_loss=1.1004


Final epoch 03/7 | train_loss=1.0013


Final epoch 04/7 | train_loss=0.9615


Final epoch 05/7 | train_loss=0.9127


Final epoch 06/7 | train_loss=0.8725


Final epoch 07/7 | train_loss=0.8511
Saved final checkpoint to: c:\Users\kipra\OneDrive\Dokumente\New project\checkpoints\unet_final_presentation.pt
FINAL OPENIMAGES-100 | loss=2.2296 | pixAcc=0.552 | mIoU_fg=0.114
background | IoU=0.570 | Precision=0.891 | Recall=0.612 | F1=0.726
    person | IoU=0.043 | Precision=0.044 | Recall=0.618 | F1=0.082
       car | IoU=0.039 | Precision=0.432 | Recall=0.041 | F1=0.076
       dog | IoU=0.260 | Precision=0.368 | Recall=0.469 | F1=0.413


## Load saved weights and predict on new images

To use during the presentation for the selected images. Load the saved checkpoint, and visualize the predicted mask.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode

CHECKPOINT_TO_LOAD = CHECKPOINT_DIR / "unet_final_presentation.pt"
PRESENTATION_IMAGE_PATHS = []

checkpoint = torch.load(CHECKPOINT_TO_LOAD, map_location=device)
loaded_features = checkpoint.get("features", [32, 64, 128, 256, 512])
loaded_image_size = tuple(checkpoint.get("image_size", IMAGE_SIZE))
loaded_class_names = checkpoint.get("class_names", CLASS_NAMES)

inference_model = UNet(
    in_channels=3,
    out_channels=len(loaded_class_names),
    features=loaded_features,
).to(device)
inference_model.load_state_dict(checkpoint["model_state_dict"])
inference_model.eval()

color_map = np.array(
    [
        [0, 0, 0],        # background
        [255, 80, 80],    # person
        [80, 180, 255],   # car
        [80, 220, 120],   # dog
    ],
    dtype=np.uint8,
)

@torch.no_grad()
def predict_mask(image_path: Path):
    image_path = Path(image_path)
    img = Image.open(image_path).convert("RGB")
    original_size = img.size

    img_resized = TF.resize(img, loaded_image_size, interpolation=InterpolationMode.BILINEAR)
    x = TF.to_tensor(img_resized).unsqueeze(0).to(device, memory_format=torch.channels_last)
    logits = inference_model(x)
    pred = logits.argmax(dim=1).squeeze(0).cpu().to(torch.uint8)

    pred = TF.resize(
        pred.unsqueeze(0).float(),
        (original_size[1], original_size[0]),
        interpolation=InterpolationMode.NEAREST,
    ).squeeze(0).to(torch.uint8)
    return img, pred.numpy()

def show_prediction(image_path: Path):
    img, pred = predict_mask(image_path)
    pred_rgb = color_map[pred]

    img_np = np.array(img).astype(np.float32)
    overlay = (0.6 * img_np + 0.4 * pred_rgb.astype(np.float32)).clip(0, 255).astype(np.uint8)

    fig, ax = plt.subplots(1, 3, figsize=(14, 5))
    ax[0].imshow(img)
    ax[0].set_title("Image")
    ax[0].axis("off")

    ax[1].imshow(pred, vmin=0, vmax=len(loaded_class_names) - 1, cmap="tab10")
    ax[1].set_title("Predicted mask")
    ax[1].axis("off")

    ax[2].imshow(overlay)
    ax[2].set_title("Overlay")
    ax[2].axis("off")

    plt.tight_layout()
    plt.show()

    present_classes = [loaded_class_names[i] for i in np.unique(pred)]
    print("Predicted classes:", present_classes)

if not PRESENTATION_IMAGE_PATHS:
    print("Set PRESENTATION_IMAGE_PATHS to one or more new image files before running this cell.")
else:
    for image_path in PRESENTATION_IMAGE_PATHS:
        print("Running inference on:", image_path)
        show_prediction(image_path)


Set PRESENTATION_IMAGE_PATHS to one or more new image files before running this cell.
